# 6 · Enhancer redundancy — Steps 1 & 2 + classification (Spark, cluster)

Enhancer sets differ per cell line (identical genes). **Step 2 recomputes the true 3D distance** of each gene's c1-nearest enhancer **in c2's model** (the spec's heavy step) — no overlap proxy.

- **Step 1 (Spark):** nearest **active** enhancer per gene per cell line (+ gene & enhancer coords).
- **Step 2 (driver, numpy via `install_deps`):** load each c2 chromosome ensemble from `model-repository`, map the c1-nearest enhancer's centre and the gene TSS to model bins `(pos − first_bin)//resolution + 1`, and compute the ensemble-mean Euclidean distance → `dist_L1_in_c2` (near-full coverage).
- **Classify (pandas):** per-chromosome tertiles; `nearest_changed` = L1/L2 don't overlap within padding; redundancy flags (toggle `REQUIRE_SMALL_IN_C1`).

Writes a portable CSV per directed comparison to `s3://database/enhancer_redundancy/<comparison>/data.csv`. Notebook 7 (local, no storage) does the DESeq2 + log2FC analysis.

In [ ]:
%%configure -f
{"executorMemory": "12G", "executorCores": 12, "ttl": "12h", "heartbeatTimeoutInSecond": 43200, "numExecutors": 3}

In [ ]:
# Install Python deps into the live Livy kernel (driver): pandas/numpy for the
# classification, s3fs to read 3D models + write the CSV outputs.
def install_deps(deps):
    from pyspark import SparkFiles
    from subprocess import call
    import sys
    for package in deps:
        call([sys.executable, '-m', 'pip', 'install', '-q', '-t', SparkFiles.getRootDirectory(), package])

install_deps(['pandas', 'numpy', 's3fs'])

import sys
from pyspark import SparkFiles
_root = SparkFiles.getRootDirectory()
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import io, json, gc
import numpy as np
import pandas as pd
import s3fs
from pyspark.sql import Window
import pyspark.sql.functions as F

ACTIVE_STATES = ['TssA', 'TssAFlnk', 'TxFlnk', 'Tx', 'TxWk',
                 'EnhG', 'EnhG1', 'EnhG2', 'Enh', 'EnhA1', 'EnhA2']
QUERY_IDS = {
    "GM12878": "a1fc46a9-93f8-424f-b41d-37bfd85d3b94",
    "H1ESC":   "f7bc6dac-6aa3-49e6-a2e5-c2ff27824c81",
    "HFFC6":   "f648f805-c3a9-4cf4-a108-94e6f5fa96c1",
}
USED_PROJECTS = ['whole_all_vs_all_gm12878_fix',
                 'whole_all_vs_all_h1esc_fix',
                 'whole_all_vs_all_hffc6_fix']
COMPARISONS = [("GM12878","H1ESC"), ("H1ESC","GM12878"),
               ("H1ESC","HFFC6"),   ("HFFC6","H1ESC"),
               ("GM12878","HFFC6"), ("HFFC6","GM12878")]

MODEL_REPOSITORY = "model-repository"   # s3 bucket: <ensemble_id>.coordinates.npy + .metadata.json
OUTPUT_BASE = "s3://database/enhancer_redundancy"
PADDING = 5000              # bp tolerance for the nearest_changed overlap test (Cond 1)
REQUIRE_SMALL_IN_C1 = True  # redundancy requires old enhancer was small(close) in c1 AND large(far) in c2.
                            # set False for the literal spec (just large in c2).

In [ ]:
def read_results(cell_line, query_id):
    return (spark.read.parquet(f"s3a://database/results/{query_id}")
            .withColumn("cell_line", F.lit(cell_line)))

results = None
for cl, qid in QUERY_IDS.items():
    df = read_results(cl, qid)
    results = df if results is None else results.union(df)

results = (results
           .where("avg_dist > 0 AND var_dist > 0")
           .where(F.col('project_id').isin(USED_PROJECTS)))

chromatin_states_df = (spark.read.parquet("s3a://database/chromatin_states")
                       .where(F.col('name').isin(ACTIVE_STATES)))
results.createOrReplaceTempView("results")
chromatin_states_df.createOrReplaceTempView("chromatin_states")

In [ ]:
# active (gene & enhancer both overlap an active ChromHMM state), like notebook 1.
# carry gene coords + strand (needed to map the gene TSS to a model bin in Step 2).
active_pairs = spark.sql("""
SELECT r.gene_id, r.gene_chr, r.gene_start, r.gene_end, r.gene_strand,
       r.enh_id, r.enh_chr, r.enh_start, r.enh_end, r.avg_dist, r.cell_line
FROM results r
WHERE EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.gene_chr
                AND cs.start <= r.gene_end AND cs.end >= r.gene_start)
  AND EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.enh_chr
                AND cs.start <= r.enh_end AND cs.end >= r.enh_start)
""")

In [ ]:
# Step 1: nearest active enhancer per (cell_line, gene); collect (small) to the driver
w = Window.partitionBy('cell_line', 'gene_id').orderBy(F.col('avg_dist').asc())
nearest = (active_pairs
           .withColumn('rk', F.row_number().over(w))
           .where('rk = 1')
           .drop('rk'))

nearest_pd = nearest.toPandas()
for c in ['gene_chr', 'enh_chr', 'gene_strand']:
    nearest_pd[c] = nearest_pd[c].astype(str)
print(nearest_pd.shape, nearest_pd.cell_line.value_counts().to_dict())

In [ ]:
# --- Step 2: TRUE 3D distance of each gene's c1-nearest enhancer (L1) in c2's model ---

# s3fs configured from the Spark hadoop conf (reuses the cluster's S3 credentials)
_hconf = spark._jsc.hadoopConfiguration()
def _hc(k, d=None):
    v = _hconf.get(k)
    return v if v is not None else d
_ep = _hc("fs.s3a.endpoint")
if _ep and not _ep.startswith("http"):
    _ssl = _hc("fs.s3a.connection.ssl.enabled", "true").lower() in ("true", "1")
    _ep = ("https://" if _ssl else "http://") + _ep
STORAGE_OPTIONS = {}
if _hc("fs.s3a.access.key"): STORAGE_OPTIONS["key"] = _hc("fs.s3a.access.key")
if _hc("fs.s3a.secret.key"): STORAGE_OPTIONS["secret"] = _hc("fs.s3a.secret.key")
if _ep: STORAGE_OPTIONS["client_kwargs"] = {"endpoint_url": _ep}
if _hc("fs.s3a.path.style.access", "true").lower() in ("true", "1"):
    STORAGE_OPTIONS["config_kwargs"] = {"s3": {"addressing_style": "path"}}
S3 = s3fs.S3FileSystem(**STORAGE_OPTIONS)
print("S3 endpoint:", _ep, "| creds:", bool(STORAGE_OPTIONS.get("key")))

# (cell_line, chrom) -> ensemble_id, from project_configuration
proj = (spark.read.json("s3a://database/project_configuration", multiLine=True)
        .select(F.col('project_id'), F.explode('datasets').alias('d'))
        .select(F.col('d.metadata.cell_line').alias('cell_line'),
                F.col('d.ensemble_region.chromosome').alias('chrom'),
                F.col('d.ensemble_id').alias('ensemble_id'),
                F.col('project_id'))
        .where(F.col('project_id').isin(USED_PROJECTS))
        .select('cell_line', 'chrom', 'ensemble_id').distinct())
ENS_BY = {(r.cell_line, str(r.chrom)): r.ensemble_id for r in proj.toPandas().itertuples()}
print(len(ENS_BY), "ensembles mapped")

def load_ensemble(ensemble_id):
    # -> (coords (n_models, n_bins, 3), first_bin, last_bin, resolution). packed.py:9 convention.
    base = f"{MODEL_REPOSITORY}/{ensemble_id}"
    with S3.open(f"{base}.metadata.json", "r") as fh:
        meta = json.load(fh)
    with S3.open(f"{base}.coordinates.npy", "rb") as fh:
        coords = np.load(fh)
    return coords, int(meta['first_bin']), int(meta['last_bin']), int(meta['resolution'])

In [ ]:
# Loop (c2, chromosome): load each ensemble ONCE, compute every comparison's genes on
# that chromosome, then release it (bounds driver memory to one ensemble at a time).
c2_groups = {}
for c1, c2 in COMPARISONS:
    c2_groups.setdefault(c2, []).append(c1)

parts = {f"{c1.lower()}_vs_{c2.lower()}": [] for c1, c2 in COMPARISONS}
for c2, c1_list in c2_groups.items():
    n2_genes = set(nearest_pd.loc[nearest_pd.cell_line == c2, 'gene_id'])
    chroms = sorted({k[1] for k in ENS_BY if k[0] == c2})
    for chrom in chroms:
        coords, first_bin, last_bin, res = load_ensemble(ENS_BY[(c2, chrom)])
        n_bins = coords.shape[1]
        for c1 in c1_list:
            comp = f"{c1.lower()}_vs_{c2.lower()}"
            sub = nearest_pd[(nearest_pd.cell_line == c1) & (nearest_pd.gene_chr == chrom)]
            sub = sub[sub.gene_id.isin(n2_genes)]
            if sub.empty:
                continue
            tss = np.where(sub['gene_strand'].values == '+',
                           sub['gene_start'].values, sub['gene_end'].values)
            ctr = (sub['enh_start'].values + sub['enh_end'].values) // 2          # L1 centre
            gbin = ((tss - first_bin) // res + 1).astype(int)                      # services.py:120-126
            ebin = ((ctr - first_bin) // res + 1).astype(int)
            valid = (gbin >= 0) & (gbin < n_bins) & (ebin >= 0) & (ebin < n_bins)
            d = np.full(len(sub), np.nan)
            if valid.any():
                a = coords[:, gbin[valid], :]; b = coords[:, ebin[valid], :]       # models.py:80-83
                d[valid] = np.linalg.norm(a - b, axis=2).mean(axis=0)
            parts[comp].append(pd.DataFrame({'dist_L1_in_c2': d, 'in_window': valid},
                                            index=sub['gene_id'].values))
        del coords; gc.collect()
    print("computed c2 =", c2)

l1_by_comp = {comp: (pd.concat(p) if p else pd.DataFrame(columns=['dist_L1_in_c2', 'in_window']))
              for comp, p in parts.items()}
for comp, t in l1_by_comp.items():
    cov = float(t['in_window'].mean()) if len(t) else float('nan')
    print(f"{comp}: {len(t)} genes, in-window coverage={cov:.3f}")

In [ ]:
# --- classification (pandas) + write CSV per comparison ---
def chrom_tertile_thresholds(df, dist_col, chrom_col):
    out = {}
    for chrom, grp in df.groupby(chrom_col):
        out[chrom] = (float(grp[dist_col].quantile(0.33)),
                      float(grp[dist_col].quantile(0.67)))
    return out

def proximity_categories(dist_series, chrom_series, thresholds):
    cats = []
    for val, ch in zip(dist_series, chrom_series):
        t = thresholds.get(ch)
        if t is None or pd.isna(val):
            cats.append('large')
        elif val <= t[0]:
            cats.append('small')
        elif val <= t[1]:
            cats.append('mid')
        else:
            cats.append('large')
    return cats

def build_comparison(c1, c2):
    comp = f"{c1.lower()}_vs_{c2.lower()}"
    n1 = nearest_pd[nearest_pd.cell_line == c1].set_index('gene_id')
    n2 = nearest_pd[nearest_pd.cell_line == c2].set_index('gene_id')
    th_c1 = chrom_tertile_thresholds(n1.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')
    th_c2 = chrom_tertile_thresholds(n2.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')

    genes = n1.index.intersection(n2.index)
    df = pd.DataFrame({'gene_id': list(genes)})
    df['gene_chr'] = n1.loc[genes, 'gene_chr'].values
    df['nearest_enh_c1'] = n1.loc[genes, 'enh_id'].values
    df['avg_dist_c1'] = n1.loc[genes, 'avg_dist'].values
    df['nearest_enh_c2'] = n2.loc[genes, 'enh_id'].values
    df['avg_dist_c2'] = n2.loc[genes, 'avg_dist'].values

    l1 = l1_by_comp[comp]
    df['dist_L1_in_c2'] = l1['dist_L1_in_c2'].reindex(genes).values
    df['L1_found_in_c2'] = l1['in_window'].reindex(genes).fillna(False).astype(bool).values

    # Cond 1: nearest enhancer changed = L1 and L2 do NOT overlap within padding
    a_chr = n1.loc[genes, 'enh_chr'].values; a_s = n1.loc[genes, 'enh_start'].values; a_e = n1.loc[genes, 'enh_end'].values
    b_chr = n2.loc[genes, 'enh_chr'].values; b_s = n2.loc[genes, 'enh_start'].values; b_e = n2.loc[genes, 'enh_end'].values
    overlap = (a_chr == b_chr) & (a_s - PADDING <= b_e) & (a_e + PADDING >= b_s)
    df['nearest_changed'] = ~overlap

    df['proximity_category_c1'] = proximity_categories(df['avg_dist_c1'], df['gene_chr'], th_c1)
    df['proximity_category_c2'] = proximity_categories(df['avg_dist_c2'], df['gene_chr'], th_c2)
    df['proximity_of_L1_in_c2'] = proximity_categories(df['dist_L1_in_c2'], df['gene_chr'], th_c2)

    # Cond 2: old enhancer far in c2  (+ optional small-in-c1 for the small->large shift)
    df['small_to_large'] = df['proximity_category_c1'].eq('small') & df['proximity_of_L1_in_c2'].eq('large')
    old_far = df['proximity_of_L1_in_c2'].eq('large')
    if REQUIRE_SMALL_IN_C1:
        old_far = old_far & df['proximity_category_c1'].eq('small')
    new_close = df['proximity_category_c2'].eq('small')        # Cond 3: new enhancer close
    df['is_redundancy'] = df['nearest_changed'] & old_far & new_close
    df['is_nonredundant_switcher'] = df['nearest_changed'] & old_far & ~new_close

    df.insert(0, 'comparison', comp); df.insert(1, 'c1', c1); df.insert(2, 'c2', c2)
    return df

for c1, c2 in COMPARISONS:
    comp = f"{c1.lower()}_vs_{c2.lower()}"
    t = build_comparison(c1, c2)
    denom = int(t['nearest_changed'].sum())
    freq = (int(t['is_redundancy'].sum()) / denom) if denom else float('nan')
    print(f"{c1}->{c2}: genes={len(t)}  redundancy={int(t.is_redundancy.sum())}  "
          f"frequency={freq:.4f}  coverage={t['L1_found_in_c2'].mean():.3f}")
    t.to_csv(f"{OUTPUT_BASE}/{comp}/data.csv", index=False, storage_options=STORAGE_OPTIONS)

print("done — pull", OUTPUT_BASE, "/* to data/whole_chromosomes/enhancer_redundancy/ for notebook 7")